In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Geometry-V2 neural corner synchronization N0

One controlled run trains the frozen small PyTorch embedder/extractor in-process, then evaluates validation and independent confirmation. It writes only bounded public operational metrics. `science_denominator=0`; geometry has coordinate authority only.


In [ ]:
import hashlib, json, os, pathlib, secrets, shutil, subprocess, sys
from datetime import datetime, timezone
from google.colab import userdata

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
BRANCH = 'Geometry-V2'
N0_RUNNER_EXACT = 'd82efc292db8a16f60d272635e577a4186ed866a'
RUNNER_PATH = 'experiments/run_geometry_v2_neural_corner_sync_n0.py'
SUCCESS_PREFIX = 'CEGWM_GEOMETRY_V2_N0 '
FAILURE_PREFIX = 'CEGWM_GEOMETRY_V2_N0_FAILURE '
MAX_CONTROL_BYTES = 1024
HANDOFF_FAILED = False
RUNNER_ATTEMPTED = False

def public_error_class(error):
    if isinstance(error, (ValueError, TypeError)): return 'validation_error'
    if isinstance(error, (FileExistsError, FileNotFoundError, PermissionError, OSError)): return 'filesystem_error'
    if isinstance(error, subprocess.SubprocessError): return 'subprocess_error'
    if isinstance(error, RuntimeError): return 'runtime_error'
    return 'unexpected_error'

def stop(stage, error):
    global HANDOFF_FAILED
    if not HANDOFF_FAILED:
        HANDOFF_FAILED = True
        print('CEGWM_GEOMETRY_V2_N0_HANDOFF_FAILURE ' + json.dumps({'stage': stage, 'error_class': public_error_class(error)}, sort_keys=True, separators=(',', ':')))

def git_output(repo, *args):
    return subprocess.run(['git', *args], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()


In [ ]:
repo = pathlib.Path('/content/ceg-wm-geometry-v2-n0')
execution_commit = ''
try:
    if repo.exists(): raise FileExistsError('create-only checkout already exists')
    subprocess.run(['git', 'clone', '--no-checkout', '--branch', BRANCH, REPO_URL, str(repo)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(['git', 'checkout', '--detach', N0_RUNNER_EXACT], cwd=repo, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    execution_commit = git_output(repo, 'rev-parse', 'HEAD')
    if execution_commit != N0_RUNNER_EXACT or git_output(repo, 'status', '--porcelain'): raise RuntimeError('checkout identity mismatch')
    if not (repo / RUNNER_PATH).is_file(): raise FileNotFoundError('N0 runner missing')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)], check=True)
except BaseException as error:
    stop('checkout', error)


In [ ]:
geometry_secret = None
derived_key_hex = ''
runner_env = None
control_read = control_write = None
if not HANDOFF_FAILED:
    try:
        if RUNNER_ATTEMPTED: raise RuntimeError('runner already attempted')
        RUNNER_ATTEMPTED = True
        try: geometry_secret = userdata.get('CEGWM_GEOMETRY_KEY')
        except Exception: geometry_secret = None
        if geometry_secret is None: geometry_secret = secrets.token_bytes(32)
        elif isinstance(geometry_secret, str) and len(geometry_secret.encode('utf-8')) >= 16: geometry_secret = geometry_secret.encode('utf-8')
        else: raise ValueError('CEGWM_GEOMETRY_KEY must contain at least 16 UTF-8 bytes')
        derived_key_hex = hashlib.sha256(b'CEG-WM/Geometry-V2/N0/runtime-key\x00' + geometry_secret).hexdigest()
        session_utc = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        drive_root = pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V2/N0')
        run_dir = drive_root / ('Geometry-V2-N0-' + execution_commit[:12] + '-' + session_utc)
        if run_dir.exists(): raise FileExistsError('create-only Drive run directory exists')
        runner_env = {name: value for name, value in os.environ.items() if not any(token in name.upper() for token in ('TOKEN', 'KEY', 'SECRET'))}
        runner_env['CEGWM_GEOMETRY_KEY_HEX'] = derived_key_hex
        control_read, control_write = os.pipe()
        command = [sys.executable, str(repo / RUNNER_PATH), '--repo-root', str(repo), '--expected-exact', execution_commit, '--output-root', str(run_dir), '--control-fd', str(control_write), '--device', 'auto']
        process = subprocess.Popen(command, cwd=repo, env=runner_env, pass_fds=(control_write,), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        os.close(control_write); control_write = None
        runner_rc = process.wait(timeout=7200)
        line = os.read(control_read, MAX_CONTROL_BYTES + 1)
        if len(line) > MAX_CONTROL_BYTES or not line.endswith(b'\n'): raise RuntimeError('invalid bounded control receipt')
        text = line.decode('utf-8', 'strict').strip()
        if text.startswith(SUCCESS_PREFIX): receipt_kind, receipt = 'success', json.loads(text[len(SUCCESS_PREFIX):])
        elif text.startswith(FAILURE_PREFIX): receipt_kind, receipt = 'failure', json.loads(text[len(FAILURE_PREFIX):])
        else: raise RuntimeError('unexpected control prefix')
        terminal = {'run_id': receipt.get('run_id'), 'n0_status': receipt.get('n0_status'), 'artifact_status': receipt.get('artifact_status', 'unavailable'), 'execution_exact': execution_commit, 'failure_point': receipt.get('failure_point'), 'error_class': receipt.get('error_class'), 'runner_rc': runner_rc, 'science_denominator': 0, 'drive_directory': str(run_dir)}
        print('CEGWM_GEOMETRY_V2_N0_TERMINAL ' + json.dumps(terminal, sort_keys=True, separators=(',', ':')))
        if runner_rc != 0 or receipt_kind != 'success': raise RuntimeError('N0 runner reported failure')
    except BaseException as error:
        stop('runner', error)
    finally:
        geometry_secret = None; derived_key_hex = ''
        if runner_env is not None: runner_env.pop('CEGWM_GEOMETRY_KEY_HEX', None)
        for fd in (control_read, control_write):
            if fd is not None: os.close(fd)
